# 02b — Preprocessing Pipeline

**Purpose:** Fit a preprocessing pipeline on the TRAINING set only, then apply it
to both train and test. Fitting on test data is data leakage.

Pipeline steps (configurable in `config/features.yaml → preprocessing`):
1. **Imputation** — fill missing values (median / mean / constant)
2. **Scaling** — standardise numeric features (standard / minmax / robust / none)
3. **Encoding** — encode categoricals:
   - `ordinal` — sklearn OrdinalEncoder (default, works with all tree models)
   - `onehot`  — sklearn OneHotEncoder (for linear/regularised models)
   - `target`  — category_encoders TargetEncoder (high cardinality)
   - `binary`  — category_encoders BinaryEncoder (compact)

To switch strategy: edit `config/features.yaml → preprocessing.categorical_encoder`.

---
**Inputs:** `data/raw/train.parquet`, `data/raw/test.parquet`  
**Outputs:** `data/processed/X_train.parquet`, `data/processed/X_test.parquet`,  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;`data/processed/y_train.parquet`, `data/processed/y_test.parquet`,  
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;`models/preprocessing_pipeline.pkl`

In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_utils import load_data, save_data, load_config, DATA_RAW, DATA_PROCESSED
from src.preprocessing import (
    build_pipeline, fit_pipeline, transform,
    get_preprocessing_summary, save_pipeline,
    compare_encoding_strategies,
)
from src.feature_engineering import _compute_derived_features

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.float_format', '{:.4f}'.format)
print('Ready.')

In [ ]:
cfg = load_config()
DURATION_COL = cfg['target']['duration_col']
EVENT_COL    = cfg['target']['event_col']

# ── Load split data ───────────────────────────────────────────────────────────
# Try to load the split files from notebook 01b first.
# Fallback: load the full dataset and split here.
try:
    df_train = load_data('train', stage='raw')
    df_test  = load_data('test',  stage='raw')
    print('Loaded pre-split data from notebook 01b.')
except FileNotFoundError:
    from sklearn.model_selection import train_test_split
    df_full = load_data('survival_data', stage='raw')
    df_train, df_test = train_test_split(
        df_full, test_size=cfg['experiment']['test_size'],
        random_state=cfg['experiment']['random_state'],
        stratify=df_full[EVENT_COL],
    )
    df_train = df_train.reset_index(drop=True)
    df_test  = df_test.reset_index(drop=True)
    print('train.parquet not found — performed stratified split as fallback.')

print(f'Train: {df_train.shape}  |  Test: {df_test.shape}')

## 1. Separate Targets and Compute Derived Features

Derived features are computed BEFORE preprocessing but AFTER splitting.
This ensures no information from the test set influences derived feature formulas.

In [ ]:
drop_cols = [DURATION_COL, EVENT_COL, 'cohort_date']

# Targets
y_train = df_train[[DURATION_COL, EVENT_COL]].copy()
y_test  = df_test[[DURATION_COL, EVENT_COL]].copy()

# Feature DataFrames (drop targets + date)
X_raw_train = df_train.drop(columns=drop_cols, errors='ignore').copy()
X_raw_test  = df_test.drop(columns=drop_cols, errors='ignore').copy()

# Compute derived features independently in each split
X_raw_train = _compute_derived_features(X_raw_train)
X_raw_test  = _compute_derived_features(X_raw_test)

print(f'X_raw_train: {X_raw_train.shape}')
print(f'X_raw_test : {X_raw_test.shape}')
print(f'\nMissing values before preprocessing:')
missing = X_raw_train.isnull().sum()
print(missing[missing > 0].to_string() or '  (none)')

## 2. Current Preprocessing Configuration

In [ ]:
pp_cfg = cfg.get('preprocessing', {})
print('Active preprocessing configuration (from config/features.yaml):')
print(f"  Numeric imputer       : {pp_cfg.get('numeric_imputer', 'median')}")
print(f"  Scaler                : {pp_cfg.get('scaler', 'standard')}")
print(f"  Categorical encoder   : {pp_cfg.get('categorical_encoder', 'ordinal')}")
print()
print('To change strategy: edit config/features.yaml → preprocessing section,'
      ' then re-run this notebook.')

## 3. Compare Encoding Strategies (optional)

Run this cell to see how each encoder affects the output shape and columns.
Comment it out in production once you've chosen a strategy.

In [ ]:
strategies_to_compare = ['ordinal', 'onehot']  # add 'target', 'binary' if category_encoders is installed

comparison = compare_encoding_strategies(
    X_raw_train,
    y_train[EVENT_COL],
    cfg,
    strategies=strategies_to_compare,
)
print('Encoding strategy comparison:')
display(comparison)

## 4. Build and Fit Pipeline on Training Data

In [ ]:
# Build the pipeline from config
pipeline = build_pipeline(cfg)

# ── FIT ONLY ON TRAINING DATA ──────────────────────────────────────────────
# This is the critical leakage-prevention step. The pipeline learns:
#   - median values (for imputation) from train only
#   - mean/std (for scaling) from train only
#   - category mappings from train only
X_train_proc, fitted_pipeline = fit_pipeline(
    pipeline,
    X_raw_train,
    y_train[EVENT_COL],
    cfg,
)

print(f'\nX_train after preprocessing: {X_train_proc.shape}')
print(f'Columns: {list(X_train_proc.columns)}')

## 5. Apply to Test Set (no refitting)

In [ ]:
# ── TRANSFORM TEST USING FITTED PIPELINE — never refit ───────────────────────
X_test_proc = transform(fitted_pipeline, X_raw_test, cfg)

print(f'X_test after preprocessing: {X_test_proc.shape}')
assert list(X_train_proc.columns) == list(X_test_proc.columns), \
    'Column mismatch between train and test!'
print('✓ Column order consistent between train and test')

## 6. Preprocessing Summary

In [ ]:
summary = get_preprocessing_summary(fitted_pipeline, cfg)
print('Pipeline summary:')
display(summary)

In [ ]:
# Verify no NaN survived
for name, X in [('train', X_train_proc), ('test', X_test_proc)]:
    nan_count = X.isnull().sum().sum()
    if nan_count > 0:
        print(f'⚠️  {nan_count} NaN remaining in {name} after preprocessing!')
    else:
        print(f'✓ {name}: zero NaN after preprocessing')

## 7. Visual: Before vs After Preprocessing

In [ ]:
# Show scaled distributions for numeric features
num_cols_cfg = [c for c in cfg.get('numeric_features', []) if c in X_train_proc.columns]
show_cols = num_cols_cfg[:6]  # show first 6

if show_cols:
    fig, axes = plt.subplots(2, len(show_cols), figsize=(3.5 * len(show_cols), 6))
    for i, col in enumerate(show_cols):
        # Before
        if col in X_raw_train.columns:
            axes[0, i].hist(X_raw_train[col].dropna(), bins=25, color='#90CAF9', edgecolor='white')
            axes[0, i].set_title(f'{col}\n(raw)', fontsize=8)
        # After
        axes[1, i].hist(X_train_proc[col], bins=25, color='steelblue', edgecolor='white')
        axes[1, i].set_title(f'{col}\n(scaled)', fontsize=8)

    axes[0, 0].set_ylabel('Raw')
    axes[1, 0].set_ylabel('After preprocessing')
    plt.suptitle('Feature distributions: raw vs preprocessed (train)', fontsize=11)
    plt.tight_layout()
    plt.show()

In [ ]:
# Correlation matrix of processed features
corr = X_train_proc.corr()
fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=ax, linewidths=0.3, annot_kws={'size': 7})
ax.set_title('Preprocessed Feature Correlation Matrix', fontsize=11)
plt.tight_layout()
plt.show()

## 8. Save Outputs

In [ ]:
# Preprocessed feature matrices
save_data(X_train_proc.reset_index(drop=True), 'X_train', stage='processed')
save_data(X_test_proc.reset_index(drop=True),  'X_test',  stage='processed')

# Targets (untouched — never scale duration or event)
save_data(y_train.reset_index(drop=True), 'y_train', stage='processed')
save_data(y_test.reset_index(drop=True),  'y_test',  stage='processed')

# Fitted pipeline (for inference on new data)
save_pipeline(fitted_pipeline, name='preprocessing_pipeline')

print('\n✓ Done. Proceed to 03_feature_engineering.ipynb or 04_feature_selection.ipynb')